# 📋 STR01 · 미션 01 | 부대별 5대 핵심 보급품(식량, 피복, 유류, 탄약, 수리부속) 재고 및 기준 보급일수(DOS) 스키마 정의

**학생용 Student · 직접 실습**  
CMS 편성: 2시간 · 교육용 가상 데이터 사용

---

## 📖 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **부대별 5대 핵심 보급품(식량, 피복, 유류, 탄약, 수리부속) 재고 및 기준 보급일수(DOS) 스키마 정의** 과제를 수행합니다.

실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 교육용 가상 부대 20개의 일일 재고·소모·보급 원장을 만들어 필드, 형식, 허용값, 의미를 데이터 딕셔너리로 정리한다.

DOS = 재고 / 최근 7일 평균 소모량, 목표 대비 편차와 달성률을 KPI로 정의하고, 분석에 쓸 피처 목록을 20개 이상 만든다.

부대 보고 수신부터 처장 승인까지 현행 KPI 보고 업무를 11단계로 그려 AS-IS/TO-BE 소요시간을 비교한다.

**만들 결과물:** `docs/str01_logistics_domain_spec.md`, `data/data_dictionary.csv`, `outputs/M01/proc.csv`, `forms/STR01_M01_forms.md`

## 📌 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

> **보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다.

- 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다.
- 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다.
- 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요.
- 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'STR01_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 2개: STR01_M01_forms.md, STR01_M01_lesson.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAAIAAAAIVzoqjBkYQkAAPUVAAASAAAAU1RSMDFfTTAxX2Zvcm1zLm1k3VhdbxvHFX3nrxjALw4qiZI/gkRAURhwAwht0qJ2HoqiEAhJMQTYliEy'
    'AQLogZKWBi3SNeVwLVJesmRCWVJKwyt+iCtUagH9FOdtZ/Y/9Nw7M+SKshOgSIEisExyZ2Zn5t577rln5oq4c/dP0zPiU/xX9W3lHAn1YlsVauJttizkcVYW'
    's7LriJv4FpHbV4W2kN1eGDyJnuevYpz8+5MJEZVPZbc/IZTXkHsVPG86yj3Dc74iX7Uxi3r89APM3w67DSH9ZyIMfNXKmplU7RQDr97+wx2M2WpFGy25nxeq'
    '4apaJZFQL0uyG4jo2ZPQz86Kv0Sug9F/HY08H0RlRzY95QRoE3K/L3jzaicn2z0x3qlqvaGlVUd+V5SlCm0LNqhORcjvD2WzJq5Nh74nMFZtrvMbvBsMVTt5'
    '5Z3KQ39ChH0fOxGq0BpfR1Wzgqw6qUQuTy6PHeXUaPJo24Md+JWLqm7k0sNUIiG72bBzJsLuqWp4s2acgEfk3qmAp9EA71fkBtYvKceDE8vhv8vid3+cE9Hf'
    'AuUfnA9koQ2j5F5D1Rzs14leOkKeOGG/FLmHspCXhdYUu3HQEOpxUX5/QBuZxj6FdIt2MVV3YCj+RLTucwCbnsBmVMHDBtQ+DHO35Bbc26HYinDgqZMqLXlr'
    'jswO++2o4hizYVMI8KiNtmxmaaKwk1WvaqP9JBJXrrAR2r80gXzqw+GR49NU8FsisSYAPsRBrGEmH5gjWw+U5yDyiMbuYeg/p756CdE8H6izsvwGSw1OwyMf'
    '7fJNoDeFrkpeVQOxhiknJyfFOz7RA0fgyf7R6sNoXGzXjh8ba4NwsVn7W0yPteqEuNRaks+qBo8zHxIOL/YflylwCC/S56OxbvLnzAzcw36/nAHsTd1LK+Vk'
    'Yw9erML5AXm1Jwsn5LKNgzAIEC0fiMQzRfTFkyTWUg7AUwBq4U3sAmH7CW9e3Pv/zxM56tYcEVG0+5Rs8aTv4ZsBSs1ggURikrCLXBJRsQS3DYmj4KlGaVa8'
    'fdGkSWShjwTWbOZT6nGHaVU7PVkvUrjM25gUqbJTAhk1ycs6SpoyemDQWVq14YXdfuQ2xlCN74YHaqWwgBiJEpCKOk70HpgSkWTCAtQ310EwppfnB3XurwPP'
    'B3Kvxzm80Y7c17MmxaJqmbCuGpxphRaWEmq3rDqvaS3ZDoSslPi1RlbV98D2DhhuNNYa/KNj7Q7BtIdlyon/Fj4JvTVCLS2ratnhnnkDREmIiGr2YvRnuMHQ'
    'rdmijTn7iThQ+hVVYd/RBFXHeEE+cwww4gS2RghQTguMPrRZr4JciTZrcWYqtJBGyLdckarXVhAjqaFbNNI4wAwHet9rgHjxfr6ChIPF3R7KLHX0PfVtzuwX'
    'bxZB5hhmfP1+x77LvZc8/M6GxPXwqDesiw4zc9OPe5htFbKTjyoBFxT/X9rJm4T5cBBgb1RG5H6Jy67LnCM3eiByzIsSZ+MIkgLfW7fF4gOEqq1+LAI6ngx2'
    'xGeLKK1YlLVTUD8VEjuAU9Mmkuw49Djs4wQd7zNl9MIOYrjX7jdxtuH6GdxO5GR0j1ZKvKxZDQRw7JHVWhLBg69JHqzZof5zGHDsILdAZ4EPqh6lqLYA7UPc'
    'cb3CC28C/CS/lYjT5XGepl6zCfA/wdJlsyfZyF5+GAjzSBnWyys3bzLahICZ0oYsFyCh5PE6pweQBZVEdgbyFexF/cNP47vY2wCQqhVJIqhchXQdNIp0Klz8'
    'uLpRRbDkCiUV9rJEF9FjEDPYddPTNQBZQMzPP+OMwS1q6wSp/UO2rMsrEgR2QRwWPBuHH7Iuh9zoa61WhjUIgXYPSGiOCjaMwLfQdZvaO/8Qllw0SYgYbEfc'
    '63thp01poiotMMhwsIXDj1LxV6n7y4upzNL8/aXFe0ur6OF/EK9WYPGCxQOmQIigrb0RJLkkuY4sVIdEQXt2jlDmzD4Z41rbxFOS+5hqxrSsdlnVg1iiE4ra'
    'd9j+sQq+ZoaQitHuMmePUU6w8SPaNJFlHP0E9tEzg990eAJZQUjM/hwHJaoCWhqyoH7fUUkvwi48BaFCol6FktZv27PRRUEty67KlaWTp7WYR0l0G61OKlxn'
    'zvtPTlbCAEIUASJ0Oo7tlhlTBpyac0YanRTU7iHlEyFCSyeK9RieSP7XcwbZRP3uGS/IldUeo2yV1KpMl8rzAeQUYld3bPB1LOhtU65f5IFJbOA3BrJr4poN'
    'GwN3VqRXF5KLKw9Syw+T6czq9My8fph69DX5c2yr8J2WNXpberNIcFA2aZ0vVlMPlq7eTmVSn9CvD8Tbx89ZZGy0DVkzXv0gCdyp/W+MxpwS+mj7Mb6Y0sA8'
    'KmgSMBt8UGRepwN5Up8INY3UISQ7ZTggSeHMVzgAdFqierRP/q8VuVm1toG988HMh3zmUzs+wIbHaT7qEl4ACb1NE18jQ8Xnn83dvXM+mLv720/xdfvWn/H5'
    '2Ty1coQ4vvYVbQOpXiHzjqydkTAzjdqwX3+Sup9eYqb1K1HV4s+sNdQPZBKLMYc1rmZRFgOs8TREoQRACEAkbYTyu+FeOKXSlsCu6FXHh6p5ZAQzuEdTjyVN'
    'XR9Q/ra9pK4ZeMWoxxF4KI1eOr9fSqdXHhIImaw0V9EKvDNdj5AZRkReht51Cz3G6i/zRsNw65AKjByBVIR81GXnhG4WtNywCOBin9Q1acj7hvZjUdBZ9jGj'
    'lhHFLBfjNpyA0YmEzKRW7y1lkMvp8wE+oJgA7QMAYsI6R5+ixbUb+gVz9uZT968uHLTFRzzC5SThi4m46I2PZICZ0sLsIkuWHHkS0qUYIr1/Xi6KF0Jrz/Fk'
    '3s3rN8Ei12+Q22t0y1JSXtPQKnkL05sp6B6HdkDXOszAWIUDpm8xYrdEWSqpyjuIvWzvKUiIH/b4yoOuXKCZjCKkZKfadTq6E+LQYfybYHihwXN3HT4w9bLK'
    '9+kmiFyDpNf3RjHnkbj7Djy2l9RCmk4YiLsuy5SYrLdsOQZ46m0Njnfk1g2TWzoPZsXiykLaEPr9lXvL6czyQtpQ+3z60dLC1IPFCQFmTyXpY35xeSGzvPIw'
    'tfr11EL6qwmx8mXm0ZeZdBJTJh+trizo1i9WVh+kk3xvOo+eeX7GTEQJmgM4GbV6/TaniZuP15pRtNBAN5urjw0Uxh0fjI+EtEZZTU7qNGjqkx/OtY5HDXYJ'
    'vSCTo1ahXJxcSsV63sRo3E//AVBLAwQUAAAACAAAACFcD+GcAsgKAABdHAAAEwAAAFNUUjAxX00wMV9sZXNzb24ubWTtWV1vG8cVfeevGMAvDkqTpj/SREAQ'
    'uLVTCK2TIHYf2qKQVEkxBNiRISoB8kZJK4MW6VqyuRbpLJllTVlSosBraWmuEBoF9FOSt53Z/9Bz78yQS5sO8tCHIg2QWORyOTP33nPOPXd5Sly7/snZgriK'
    '/38o1YR8UZLVkjxyxEX8FYnbVZUDIY/COLqbPCifVpWm/PpuViS1vjzqZoXyfLlTx/s1R7kv8b5cl08PsIq6c+8toVoH8ZEvZHBfxFGgOiWzkmr2cePpyx9d'
    'wz0bnWS1I3fLQvmuatYzmVOnhFpbSRr78VFfyG/2ky0vk4mDEi6KP348iTM6ymkKWQll5Vi1NmXbU46XOrre9qSn7lTlN3snPb2GwHaqURLxYZC49/DnJQ4j'
    'd/pmvcTdl5WyrHRy/43QEYg5Bm/5puD1Jsp5jlDVbgmnPI0I9bdtXmgBeS9QzTBxAiFrrlqvSadMe+HwOmmJW6dEc3awCV6q7XV5EOKwDqcnwnJC7nZVk9Kz'
    'hR0T16Ngdbq/2pRHEUWQPLor5GFNeQ5tm9y/i7xnMmfsHYnr4CYhn0VqvZ7D5o46DIWqdNRGV21vogxyo2bWGSZ08H2UZK0s/rCwLJJ/RjiDKOQuqooXB86P'
    'pRqtErXj4AEiFcp1ZKX+Y8kd7q1XnRBxd1N5bfV4XxhM6MKLc2fjwOPMIcHN/uso4OzTaqq1Q3mQu1X5sKMehRSWfOihlHUXVcbf7TLWx1FQ1WYd0TJMxteg'
    '7VEN8MoklGAm3rPQzwvV9eJeX/yWjkT57KE2fCIGk4EmAffY4awEe1xwRoXc8emgKKvZZlDqrEEtki7Uw4hAqQ7rtJxsN+k7nA2cPuQMUaQ+n84ky6KkXFcV'
    'H9coLCyAzAB4x6oZxcdVABIZcQgSrwGLElIoyMpefEQAE3Evkn5HXLp2ZvJa/vpHZ353haJUj2u6vJzuYwels7A7I/6mwfT3IdJPeq/CVbAovRHLocGyUA1H'
    'PqnKTWbBSC5GEsHf4CziVoUie325H2QBqYBgDQS+ug/VgvB0bClmxYfrBgDi1XrScBEW3uRSaF0N1AtPHvQnxNzibDFfXF46W5i6uXhjobi8MFucmlu8NbPw'
    '2VTx9vxs7tYcS+R4gGmC+l581BUkdk+qxEe8S+50B+G7ZbkanvR+v3hz5h8UBtXMXtTsFHI9SjYi+WLFXjFh+k1cFn+d/JgP/7IGIkyI6eLSbF4f0Zxcv8nd'
    '/nI6Z6UxcfdI84CEuHsAoGBhvwQEmUypw2/tUZKvHHFpknP5osZJOypBf0cUYhD9hBg52fTczPJM/pMrly5fvYJMTTMV9/rYCoV5iTjMEprX71KxNZ3B+FQa'
    'C4WRT0C/eyhSa136O4W38REOr4+GswLp7+CSiQwVH5Wyioe0C/VoC1qBVHEDnUIDnfp0celWkc7I5eSyL83fXlxaLpIskWg2HC6XrlpSAWB0eZuhPCwbWaV7'
    'TYZ1XqneSbUKGP50Yeg2Lglu+2Lm5gISNz91c37uxvzSNEOG+tSEEUDx6dLMrfnTl5HcD+jVW+KHOw/QFEtqlSr7HbGHWmkQ5ePngdp9CEQfQO9zpge8a1OJ'
    'dgrZRvo8n0nwANlljcmjaWIVfAL+tDZNW8kT/8p1fA89EdJExNwl6DSrfFl1ttBdT3qFt8+yiG8HaKd4e5ZpTDpBVeFjWsE15Pjzh5PXr530Jq9fuYo/ly/9'
    'Bf9+OEVXuenh9I/37VdMH1PboZBlRwJHzZK9qAN774OZm8V5kjYZ1JOGpf8IEU1IjEinzp2PtVMzzPescCl/BQhCyekgEAukAMuRvCp3A1+zDAjqCD0lA2qt'
    'yTjRPKPlze7MUgKvMSz2OhSOcqgjsEpHJiGIVBuuJxRXCwSvV9AxpPAI0EdAyBxmSpNZ6Rpms64fhaodmvxQVdvhKM0FmRuf6WSdAeXEoRVtUxh4L5O5S5PU'
    'E5NGjfp2hJyiw/QBRCp9coe6eh1IqHMD811GEdkxFGJ1Lz5esUEQuRK3QfqXNDx5v0FXTpnX2vGCFsnje2Q1NcBBtU30tUzGCLlWesYr6zrnHvszAjim5kuK'
    'nNo+pALKQpEDvCGOK1etDRQMZ+lW4VpW8B/0I5SPvo8PqwLLQ1MS10fPfZ9W1CdIJQXaiT4vd9txFBnQ2ROQ6HrgNaoOYCFPMtiXhy5DnmC2n0qFDfwcB24E'
    'meMxlYLN1k5VX+XouM62uocBEsVpNkXHNx90zFL4LjJJO2vIsDfxV5h2g1BGIDEhhtacA2fFQMXqSJz2k4inFv+7xg5E2yMgIm2O3gSkEa9lSqILYDZTLdgS'
    'rbYrAbt3kN3WIsVO8kVoIACgp44btKUJ1zDGZAYgYJWR7RIbIkjU0+b4/J/XwNOABszBTS5F60BnG9gzsB6WYaSz2hvT4WrTo7uXOHeBTU9ny/KJwHK/YVwJ'
    'z1XPoiGx7SC0WpflECEjXe8TBZXTgb0BxDRIlOMna83z8fNw4KccMYAhn4iX5oisXwJQGihPpYvKy/v0qoMMC43W8fm5oF1f11P/Wjdk0Pw0lmrATT2fmPGK'
    'lIKxZRAxYOoYRSEnS8MKZ9bDqmkFgpv/+q5x4FluylEbF/HyICJE6vZNJwLi4rBkVJ+H1rUR1xCOBsFqOUafQjhKaqSEGKGe9AmW/iZHSZ065Wl4NEi5GsG2'
    'BsPT8szSjfllmIHiSQ//IDNoj3uYn7LjkEHTlgYEe5/fjFgf8Q7f4Y6zP2LkTiaHzq7Gktz0jKfiRWSzT7dI7/vBSNcY4jVl5u0kQeFdPH8RTuT8BTLazdIr'
    '0x462rDrEY/pBERrJBcja5MnFDtTDVWiRPKgvL10y2RdYJ2W+yGrt8YXRLxMIVI30C1loAnMPdz/LEqJOnU5h4ANJKggsM8VaF5j3RiFuIEbw5lnFgNe8njT'
    'P2NQmOb7yBHTP1NzC7PLC4ufzSx9mZstfsEfLn6+fPvz5WIehjR/e2lx1n7A5tRgF9zJpjotjWmH35EH2wJPswzaENPYiAplrZlHaJj2gFeMfE/71H9SApye'
    'gDSPdHzkJXX3YAhzn+SeiuEQf7m0WXYscEn+iiGM0UhaVFONx3uDODYCKdLpMrD2olW30w90+MFAZKdEIj0zlsa/MZo/nCkyGePPmcVsROpGoygg03FtBQs5'
    'foymDcXE/9VTJGr1ZJ8f18gSaAkYllezE0hhD05Elhs7PEf5m8SEV+woyV9rnaQeOqKnFt6Qm9GgrZAppDvZtGm4nPTQmSEDLceqqa4FWxKNiUdl094y50y1'
    'WLgmxJvnqvGGOd2Q9RlJVH8drf4HRyv0MnyqXuyr9nOylmzpvtVzjNY05NWIGriT18KGrxiTO8AMZRRS96f5YnHxM8IeN0PtOmgHPpkZzrY3EZyqr6QQd94g'
    'jpE58Yt8/mXEeUB8o/OwgzCNuqEck88GX3WluPDcbvK62wzmSmPoU8n/1Q39ktwQGw943dZOfjhXoO665RMfzaSvJwiad+3UM6TUBU0pDf+f9aQ1K97knrJi'
    'nHXKauOUf/0pHwmAZjxzEAdkl6BlmicNrR9UimOSX45SP6mg6m0H0Hfw0OKbfmaoHOC7duDASx446ILdQm/IUog9qKlRK3KJga2yKc0wPZnUzy4/+atL3jz0'
    '5x9fBr+95MzjHD2ss0EgoyFXQ/p1hC+ShOonIRvQa5/cx9Dv/gdQSwECFAMUAAAACAAAACFc6KowZGEJAAD1FQAAEgAAAAAAAAAAAAAApAEAAAAAU1RSMDFf'
    'TTAxX2Zvcm1zLm1kUEsBAhQDFAAAAAgAAAAhXA/hnALICgAAXRwAABMAAAAAAAAAAAAAAKQBkQkAAFNUUjAxX00wMV9sZXNzb24ubWRQSwUGAAAAAAIAAgCB'
    'AAAAihQAAAAA'
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 01. 부대별 5대 핵심 보급품(식량, 피복, 유류, 탄약, 수리부속) 재고 및 기준 보급일수(DOS) 스키마 정의

**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 5대 핵심 보급품(식량, 유류, 탄약 등) 재고 및 기준 보급일수(DOS) 스키마를 정의한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **5대 핵심 보급품(식량, 피복, 유류, 탄약, 수리부속)의 재고와 기준 보급일수(DOS) 핵심성과지표(KPI)의 스키마와 데이터 딕셔너리를 정의하고 KPI 보고 업무 프로세스 맵을 작성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **교육용 가상 부대 20개의 일일 재고·소모·보급 원장을 만들어 필드, 형식, 허용값, 의미를 데이터 딕셔너리로 정리하고 규격화하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/str01_logistics_domain_spec.md 및 데이터 딕셔너리` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 실무 업무 프로세스와의 일치성 및 분석 목표의 명확성

#### 미션 PASS 조건

- 정량 기준: 업무 프로세스 맵 작성 완료, 필요 피처(Feature) 목록 20개 이상 정의

- 진행 기준: 도메인 분석서 및 데이터 요구명세서 승인

- 검증 방법: 군수 담당 실무자 인터뷰 및 기존 전산체계 데이터 구조 대조

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


### AI 활용 예시 · 미션 01 전용

> 나는 Python을 처음 배우는 학습자야. **부대별 5대 핵심 보급품(식량, 피복, 유류, 탄약, 수리부속) 재고 및 기준 보급일수(DOS) 스키마 정의**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: src/domain/str01_domain.py / validate_ledger: 원장 frame(DataFrame) → 검사 항목별 참/거짓 사전. 필수9필드·결측·유한값·부대/품목·일자 범위/정수·기상 이진·음수·중복·160일 연속·100개 키를 검사한다. 제공 UNITS·ITEMS·DAYS·N_UNIT을 사용한다. 필수 열 누락은 필수 필드=False로 반환하고 제공 자료 범위를 넘는 승인·실제 업무 적합성을 확정하지 않는다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.


In [ ]:
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """부대별 핵심 보급품 5종(식량·피복·유류·탄약·수리부속)의 재고와 기준 보급일수(DOS) 핵심성과지표 스키마와 데이터 딕셔너리를 정의하고, 피처 목록 20개 이상과 KPI 보고 업무 프로세스 맵(AS-IS/TO-BE)을 작성한다."""
    import numpy as np
    import pandas as pd

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
    # 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
    # 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
    # ---------------------------------------------------------------------------
    ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
    N_UNIT, DAYS = 20, 160
    UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


    def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
        """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
        rng = np.random.default_rng(seed)
        rows = []
        for u in range(1, n_unit + 1):
            s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
            surge = np.zeros(days)
            for st in rng.choice(days - 10, 4, replace=False):
                surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
            tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
            plan = tempo + rng.normal(0, 0.08, days)
            wx = (rng.random(days) < 0.15).astype(int)
            for it, (b, sens, tg) in ITEMS.items():
                stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
                for t in range(days):
                    c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                    r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                    stock = max(stock + r - c, 0.0)
                    rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
        return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


    def make_samples(d, lo=21, hi=125):
        """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
        out = []
        for (u, it), g in d.groupby(["unit", "item"]):
            c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
            pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
            cs = np.cumsum(c)
            tg = ITEMS[it][2] * ITEMS[it][0]
            for t0 in range(lo, hi):
                idx = np.searchsorted(cs, cs[t0] + stk[t0])
                if idx >= len(cs) or idx <= t0:
                    continue
                c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
                last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
                out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                            "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                            "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                            "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                            "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
        return pd.DataFrame(out)


    def mape(a, b):
        # [TODO : 학생 실습 1] 부대별 5대 핵심 보급품(식량, 피복, 유류, 탄약, 수리부속) 재고 및 기준 보급일수(DOS) 스키마 정의 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    d = make_data()
    s = make_samples(d)

    # 1) 스키마(데이터 딕셔너리): 필드, 형식, 허용 범위, 의미
    DICT = [("unit", "str", "가상NN부대", "보고 부대"), ("item", "str", "식량/피복/유류/탄약/수리부속", "핵심 보급품 5종"),
            ("day", "int", "0~159", "기준 일자(일 단위 일련번호)"), ("tempo", "float", "0.3~2.5", "실제 작전 템포 지수(1.0=평시)"),
            ("plan", "float", "0.3~2.5", "사전 계획 템포(미리 알 수 있는 값)"), ("wx", "int", "0/1", "악천후 여부"),
            ("consume", "float", ">=0", "일일 소모량"), ("resupply", "float", ">=0", "당일 보급 입고량"), ("stock", "float", ">=0", "일 마감 재고"),
            ("target_dos", "float", "5~20일", "기준 보급일수 목표(KPI 목표, 교육용 가정)"), ("dos", "float", ">=0", "KPI = 재고 / 최근 7일 평균 소모량")]
    dd = pd.DataFrame(DICT, columns=["필드", "형식", "허용값", "의미"])
    print("[데이터 딕셔너리] 필드", len(dd), "개 [교육용가상자료]")
    print(dd.to_string(index=False))
    bad = 0
    for col, lo in (("consume", 0), ("resupply", 0), ("stock", 0)):
        bad += int((d[col] < lo).sum())
    bad += int((~d["item"].isin(ITEMS)).sum()) + int((~d["wx"].isin([0, 1])).sum())
    print(f"\n샘플 {len(d)}행({N_UNIT}개 부대 x 5종 x {DAYS}일) 스키마 검증 위반 {bad}건")
    print("품목별 목표 DOS(KPI 목표, 가정):", {k: v[2] for k, v in ITEMS.items()})

    # 2) KPI 정의
    print("\n[KPI 정의] DOS = 재고 / 최근 7일 평균 소모량, 편차 = 현재 DOS - 목표 DOS, 달성률 = DOS / 목표 DOS")
    last = d[d["day"] == 100].copy()
    last["c7"] = [d[(d["unit"] == r.unit) & (d["item"] == r.item) & (d["day"].between(94, 100))]["consume"].mean() for r in last.itertuples()]
    last["dos"] = last["stock"] / last["c7"]
    last["target"] = last["item"].map(lambda i: ITEMS[i][2]); last["gap"] = last["dos"] - last["target"]; last["rate"] = last["dos"] / last["target"]
    print("100일째 품목별 평균 DOS와 달성률:")
    print(last.groupby("item").agg(평균DOS=("dos", "mean"), 목표=("target", "first"), 평균달성률=("rate", "mean"), 미달부대수=("rate", lambda x: int((x < 0.5).sum()))).round(2).to_string())

    # 3) 피처 목록(분석·예측에 쓸 후보): 자동 생성 열 + 추가 후보
    feat_auto = [c for c in s.columns if c not in ("unit", "item", "t0", "H")]
    extra = ["요일 효과", "월말 보급 주기 여부", "수송 거리(가상)", "기후 지수", "부대 규모(가상)", "최근 긴급 불출 횟수", "정비 지시서 건수", "훈련 일정 밀도"]
    feats = feat_auto + extra
    print(f"\n[피처 목록] 자동 산출 {len(feat_auto)}개 + 추가 후보 {len(extra)}개 = {len(feats)}개")
    print(" ", ", ".join(feats))

    # 4) KPI 보고 업무 프로세스 맵(AS-IS / TO-BE, 소요 분은 가정값)
    steps = [("부대별 재고 보고 수신", 30, "표준 형식 자동 수집", 2), ("품목별 일소모량 확인", 40, "소모율 자동 정규화", 2), ("엑셀로 DOS 재계산", 60, "KPI 자동 계산", 1),
             ("목표 대비 편차 표 작성", 45, "편차 자동 산출", 1), ("기준 미달 부대 선별", 35, "미달 규칙 자동 적용", 1), ("결핍 예상 시점 수기 추정", 70, "예측 모델 자동 산출", 3),
             ("악천후·작전 템포 반영 검토", 50, "계획·예보 피처 자동 반영", 3), ("경보 등급 판단", 30, "주의/경계/심각 자동 분류", 1),
             ("군수처 보고서 작성", 80, "대시보드 자동 갱신", 2), ("보고서 수정·재집계", 55, "단일 원천 사용", 3), ("처장 승인·조치 지시", 40, "HITL 승인 화면", 15)]
    proc = pd.DataFrame(steps, columns=["프로세스 단계", "AS-IS(분)", "TO-BE 개선", "TO-BE(분)"]); proc.index = range(1, len(proc) + 1)
    a, t = proc["AS-IS(분)"].sum(), proc["TO-BE(분)"].sum()
    print("\n[업무 프로세스 맵]")
    print(proc.to_string())
    print(f"\n프로세스 단계 {len(proc)}개, 1회 집계 AS-IS {a}분 -> TO-BE {t}분({1 - t / a:.0%} 단축 예상, 가정값)")
    print(f"목표(프로세스 맵 작성, 피처 20개 이상): 프로세스 맵 {'완료' if len(proc) >= 10 else '미달'}, 피처 {len(feats)}개 {'달성' if len(feats) >= 20 else '미달'}")


## 📋 3. 학습활동 4단계

### 활동 1: 무엇을 확인하고 계획하나요?

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, 보급품 재고 및 기준 보급일수(DOS) 계산 시 발생할 수 있는 경계 조건(예: DOS 분모가 0이거나 현재 재고가 0인 상황)에서 각각의 수치를 어떻게 직관적이고 정확하게 표시할 것인지 사전에 판단합니다.
  - 아울러 AI 에이전트에 위임할 분석의 범위, 검토할 자료의 한계, 데이터 제약 사항, 그리고 최종 결과물에 대한 확인 방법을 체계적으로 수립합니다.
- **수행 방식:**
  - 막연한 코딩이나 AI의 무분별한 답변에 의존하지 않고, 군수 데이터 분석의 정확성을 높이기 위한 명확한 사전 규칙과 검증 계획을 먼저 문서화합니다.

#### 2. 세부 수행 절차

- **원본 목표 및 자료 분석:**
  - 5대 핵심 보급품(식량, 피복, 유류, 탄약, 수리부속)의 재고와 기준 보급일수(DOS) 핵심성과지표(KPI) 정의에 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다.
  - 경계 조건(분모 0, 재고 0) 처리 원칙 수립: 일일 소모량이 0이어서 DOS 산출 시 분모가 0이 되는 경우 임의의 큰 숫자로 왜곡하지 않고 산출 불가 사유를 명시하는 규칙을 정하고, 보유 재고가 0인 상태는 별도의 관찰 경보 대상으로 분류하는 사전 기준을 마련합니다.
- **AI 위임 범위 및 검증 계획 정의:**
  - AI 에이전트에 맡길 코드 구현 및 텍스트 검토의 범위를 설정하고, 입력된 데이터의 무결성을 검증할 구체적인 확인 방법과 제약 조건을 문서( forms/STR01_M01_forms.md )에 명시합니다.

#### 3. 주의 및 검증 사항

- 입력 시점이나 기간, 분모가 서로 다른 데이터를 동일한 조건으로 잘못 해석하지 않도록 데이터 해석의 엄정성을 유지해야 합니다.
- 본 단계에서 수립된 사전 판단과 검증 계획은 다음 단계(AI와 코드 구현 및 첫…

#### 완료 확인:

- 이 활동의 확인 근거 M01 계획: 5대 핵심 보급품(식량, 피복, 유류, 탄약, 수리부속)의 재고와 기준 보급일수(DOS) 핵심성과지표(KPI)의 스키마와 데이터 딕셔너리를 정의하고 KPI 보고 업무 프로세스 맵을 작성한다.에 필요한 입력·기간·단위·사용 가능 시점과 validate_ledger의 입출력 계약을 설명하고 본인의 사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2: AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 핵심 함수인 validate_ledger 를 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다.
  - 아울러 원장 프레임워크(DataFrame)와 검사 항목 간의 검증 계약을 충족시키고, 목표 DOS(기준 보급일수) 변경에 따른 KPI 편차 및 달성률을 전후 비교하며, 평균 소모가 0일 때의 경계 사례 처리 규칙을 구현합니다.
- **수행 방식:**
  - AI와 협업하여 요구되는 데이터 검사 로직과 경계 조건 처리 규칙을 실제 코드에 반영하고, 실행 로그와 산출물을 정확하게 확보합니다.

#### 2. 세부 수행 절차

- 핵심 함수 validate_ledger 구현 및 첫 성공 실행: 제공되는 UNITS , ITEMS , DAYS , N_UNIT 변수를 활용하여 필수 9개 필드, 결측값, 유한값, 부대 및 품목, 일자 범위 및 정수, 기상 이진, 음수, 중복, 160일 연속, 100개 키 조건 등을 검사하는 검증 로직을 구현합니다.
- 필수 열이 누락된 경우 필수 필드를 False 로 반환하며, 제공된 자료 범위를 초과하는 임의 승인이나 실제 업무 적합성을 자의적으로 확정하지 않도록 계약 조건을 준수하여 첫 성공 실행을 기록합니다.
- **목표 DOS 변경에 따른 전후 비교 과제 수행:**
  - 기존 품목 하나의 목표 DOS 설정을 변경하여, 이에 따른 KPI 편차와 목표 달성률이 어떻게 달라지는지 전후 비교 분석을 코드로 구현합니다.
- **평균 소모 0 경계 조건 처리 규칙 적용:**
  - 일일 소모량이 0이 되어 DOS 산출 시 분모가 0이 발생하는 경계 상황에서 임의의 큰 숫자로 왜곡하지 않는 예외 처리 규칙을 AI와 함께 구현하고 경계 사례 데이터로 검증합니다.

#### 3. 주의 및 검증 사항

- 코드 구현 및 실행 과정에서 발생하는 실제 로그와 출력 결과를 누락 없이 저장해야 합니다.
- 단순히 AI가 생성한 코드를 무비판적으로 복…

#### 완료 확인:

- 이 활동의 확인 근거 M01 구현: src/domain/str01_domain.py의 validate_ledger를저장하고계약〔원장 frame(DataFrame) → 검사 항목별 참/거짓 사전.
- 필수9필드·결측·유한값·부대/품목·일자 범위/정수·기상 이진·음수·중복·160일 연속·100개 키를 검사한다.
- 제공 UNITS·ITEMS·DAYS·N_UNIT을 사용한다.
- 필수 열 누락은 필수 필드=False로 반환하고 제공 자료 범위를 넘는 승인·실제 업무 적합성을 확정하지 않는다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3: 원자료와 결과를 대조·수정하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 앞서 구현 및 수정된 코드를 바탕으로 저장 및 재실행을 수행하며, 24개의 전체 피처 후보 중 실제 자동 산출이 가능한 피처와 미구현 후보를 엄격하게 분류 합니다.
  - 아울러 AI의 설명 및 코드 선택 3건 이상을 원본 자료 및 실제 실행 결과와 철저히 대조하고, 보완된 답변 내용을 다시 한번 검증합니다.
- **수행 방식:**
  - 단편적인 코드 실행에 그치지 않고, 군수 데이터 분석 목표와 프로세스 맵의 정합성을 검증하며 AI의 답변을 비판적으로 교차 대조합니다.

#### 2. 세부 수행 절차

- **수정 코드 저장 및 재실행:**
  - 보완된 검증 로직 및 처리 규칙이 반영된 코드를 안전하게 저장하고 재실행하여 오류 여부와 로그를 확인합니다.
  - 피처 후보 24개 분류 (자동 16개 + 추가 후보 8개): 총 24개의 피처 후보 중 실제 자동으로 산출되는 16개 와 미구현 상태인 추가 후보 8개 를 명확히 구분하고, 추가 후보는 실제 구현된 입력 8개를 의미하는 것이 아님을 인지하여 정리합니다.
- **AI 설명·코드 선택 3건 이상 근거 대조 및 재검토:**
  - AI가 제안하거나 설명한 코드 선택 사항 중 핵심적인 3건 이상을 선정하여, 실제 군수 실무 자료와 실행 결과 데이터에 비추어 타당한지 대조하고 보완 답변을 도출합니다.

#### 3. 주의 및 검증 사항

- 프로세스 11단계와 535분에서 34분으로 단축된 수치는 교육용 가정이므로 실제 데이터 조건과 혼동하지 않도록 유의해야 합니다.
- DOS는 일(Day) 단위이고 목표 달성률은 비율이라는 점, 소모량이 0일 때는 DOS 산출 불가 사유를 표시하고 이미 재고가 0인 상태는 별도의 관찰 경보로 처리해야 하는 원칙이 정확히 반영되었는지 검증해야 합니다.

#### 완료 확인:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M01 검증: [필수] 스키마·프로세스 맵 — 업무 프로세스 맵이 작성 완료되고 피처 목록 20개 이상이 정의되었으며, 군수 실무 프로세스와 일치하고 분석 목표가 명확한가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - 원장 9개 필드와 딕셔너리 11개〔target_dos·dos 포함〕, 피처 후보 24개〔자동 16개+추가 후보 8개〕를 구분합니다.
  - 추가 후보는 실제 구현된 입력 8개라는 뜻이 아닙니다.
  - 프로세스 11단계와 535→34분은 교육용 가정입니다.
  - DOS는 일 단위이고 목표 달성률은 비율입니다.
  - 소모 0이면 DOS 산출 불가 사유를 표시하고 이미 재고 0은 별도 관찰 경보로 처리합니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 …

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4: 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 미션 1(부대별 5대 핵심 보급품 재고 및 기준 보급일수 스키마 정의) 전 과정을 수행하며 산출된 원본 평가 기준, 변경 전후의 코드 실행 결과, 실제 AI와의 대화 기록, 그리고 최종 보고서 를 누락 없이 취합합니다.
  - 아울러 정량적 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다.
- **수행 방식:**
  - 단편적인 파일 제출에 그치지 않고, 이번 미션에서 달성한 성과와 미진한 부분을 객관적으로 검수하여 후속 미션(미션 2: 전처리 모듈 구현)으로 온전히 인계할 수 있도록 패키지를 정비합니다.

#### 2. 세부 수행 절차

- **핵심 산출 파일 패키지 확정 및 제출:**
  - docs/str01_logistics_domain_spec.md (군수 도메인 명세서), data/data_dictionary.csv (데이터 딕셔너리), outputs/M01/proc.csv (처리 결과 데이터), 그리고 forms/STR01_M01_forms.md (실습 기록 서식)의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다.
- **변경 전후 코드 및 AI 대화 기록 보존:**
  - 검증 로직 구현 과정에서 수정된 코드의 변경 전후 내역과, AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 근거 대화 내용을 실습 대화 보존 위치에 명확히 연결합니다.
- **상태별 최종 판단 기록 작성:**
  - 정량적 목표 달성 여부, 현장 여건상 미처 측정하지 못한 영역, 문서 작성 완료 상태, 그리고 실제 교관 승인 대기 상태를 투명하게 구분하여 서식( forms/STR01_M01_forms.md )에 상세히 기재합니다.

#### 3. 주의 및 검증 사항

- 실제 수행 기록이나 검증 근거가 없는 항목을 마치 완료된 것처럼 임의로 작성하거나 과장하여 제출하는 행위를 엄격히 금지합니다.
- 본…

#### 완료 확인:

- 이 활동의 확인 근거 M01 완료: docs/str01_logistics_domain_spec.md, data/data_dictionary.csv, outputs/M01/proc.csv, forms/STR01_M01_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 🔍 4. 평가 기준과 확인할 자료

- [정량적 평가] [필수] 스키마·프로세스 맵 — 업무 프로세스 맵이 작성 완료되고 피처 목록 20개 이상이 정의되었으며, 군수 실무 프로세스와 일치하고 분석 목표가 명확한가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- **[정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.**
  - **PASS 증거**
    - **수행 조건과 판정 기준**
      - 원장 9개 필드와 딕셔너리 11개〔target_dos·dos 포함〕, 피처 후보 24개〔자동 16개+추가 후보 8개〕를 구분합니다.
      - 추가 후보는 실제 구현된 입력 8개라는 뜻이 아닙니다.
      - 프로세스 11단계와 535→34분은 교육용 가정입니다.
      - DOS는 일 단위이고 목표 달성률은 비율입니다.
      - 소모 0이면 DOS 산출 불가 사유를 표시하고 이미 재고 0은 별도 관찰 경보로 처리합니다.
    - **보완 방법**
      - 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- **[진행 추적 및 완료 기준] 정성·검증:**
  - **수행 조건과 판정 기준**
    - 원장 9개 필드와 딕셔너리 11개〔target_dos·dos 포함〕, 피처 후보 24개〔자동 16개+추가 후보 8개〕를 구분합니다.
    - 추가 후보는 실제 구현된 입력 8개라는 뜻이 아닙니다.
    - 프로세스 11단계와 535→34분은 교육용 가정입니다.
    - DOS는 일 단위이고 목표 달성률은 비율입니다.
    - 소모 0이면 DOS 산출 불가 사유를 표시하고 이미 재고 0은 별도 관찰 경보로 처리합니다.
  - **제출 파일 및 완료 기록**
    - 파일: docs/str01_logistics_domain_spec.md, data/data_dictionary.csv, outputs/M01/proc.csv, forms/STR01_M01_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
    - 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
  - **보완 방법**
    - 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.
**실제 출력 없이 달성으로 표시하지 않습니다.**

In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 1
        DELIVERABLE_PATHS = ['docs/str01_logistics_domain_spec.md', 'data/data_dictionary.csv', 'outputs/M01/proc.csv', 'forms/STR01_M01_forms.md']
        PROJECT_NAME = 'STR01'
        WORK_ROOT = Path(STR01_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                if state['loaded']:
                    state['dirty'] = True
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
